# 01 - Data Exploration: Karatsuba Traces for Multiplication

This notebook explores the training data for the recursive multiplication project:
- Generate and visualise Karatsuba trace examples
- Compare Karatsuba vs school algorithm trace lengths
- Visualise hierarchical position encodings
- Analyse token statistics

**Run on Colab with CPU or free T4 — no GPU needed for data exploration.**

In [ ]:
# Cell 1: Install dependencies and clone repo
import os

# Clone the repository (update URL to your repo)
REPO_URL = "https://github.com/YOUR_USERNAME/karatsuba-transformers.git"  # TODO: update
REPO_DIR = "/content/karatsuba-transformers"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

# Install the project in editable mode
!pip install -q -e {REPO_DIR}

import sys
sys.path.insert(0, REPO_DIR)

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("Setup complete.")

In [ ]:
# Cell 2: Generate sample Karatsuba traces and visualize

from src.data.karatsuba_trace import generate_karatsuba_trace
from src.data.tokenizer import KaratsubaTokenizer

# --- Helper: manual Karatsuba trace for demonstration ---
def karatsuba_trace_demo(x: int, y: int, n_bits: int, base_bits: int = 4, depth: int = 0):
    """Generate a human-readable Karatsuba trace for demonstration."""
    indent = "  " * depth
    x_bin = format(x, f'0{n_bits}b')
    y_bin = format(y, f'0{n_bits}b')
    lines = []
    lines.append(f"{indent}[MUL] {x_bin} x {y_bin}  (decimal: {x} x {y})")

    if n_bits <= base_bits:
        result = x * y
        r_bin = format(result, f'0{2*n_bits}b')
        lines.append(f"{indent}[BASE] = {r_bin}  (decimal: {result})")
        return lines, result

    half = n_bits // 2
    mask = (1 << half) - 1

    x_hi, x_lo = x >> half, x & mask
    y_hi, y_lo = y >> half, y & mask

    lines.append(f"{indent}[SPLIT] x_hi={format(x_hi, f'0{half}b')} x_lo={format(x_lo, f'0{half}b')}")
    lines.append(f"{indent}        y_hi={format(y_hi, f'0{half}b')} y_lo={format(y_lo, f'0{half}b')}")

    # z0 = x_lo * y_lo
    lines.append(f"{indent}[SUB_MUL z0] x_lo * y_lo:")
    sub_lines, z0 = karatsuba_trace_demo(x_lo, y_lo, half, base_bits, depth + 1)
    lines.extend(sub_lines)

    # z2 = x_hi * y_hi
    lines.append(f"{indent}[SUB_MUL z2] x_hi * y_hi:")
    sub_lines, z2 = karatsuba_trace_demo(x_hi, y_hi, half, base_bits, depth + 1)
    lines.extend(sub_lines)

    # z1 = (x_lo + x_hi) * (y_lo + y_hi) - z0 - z2
    sum_x = x_lo + x_hi
    sum_y = y_lo + y_hi
    lines.append(f"{indent}[ADD] sum_x = x_lo + x_hi = {sum_x}, sum_y = y_lo + y_hi = {sum_y}")

    lines.append(f"{indent}[SUB_MUL z1] sum_x * sum_y:")
    sum_bits = half + 1  # sums may need one extra bit
    sub_lines, z1_raw = karatsuba_trace_demo(sum_x, sum_y, sum_bits if sum_bits > base_bits else base_bits, base_bits, depth + 1)
    lines.extend(sub_lines)

    z1 = z1_raw - z0 - z2
    lines.append(f"{indent}[SUB] z1 = {z1_raw} - {z0} - {z2} = {z1}")

    result = (z2 << n_bits) + (z1 << half) + z0
    r_bin = format(result, f'0{2*n_bits}b')
    lines.append(f"{indent}[COMBINE] z2*B^{n_bits} + z1*B^{half} + z0 = {result}")
    lines.append(f"{indent}[RESULT] {r_bin}  (decimal: {result})")

    return lines, result

# Generate a few example traces
print("=" * 80)
print("EXAMPLE 1: 8-bit x 8-bit Karatsuba trace")
print("=" * 80)
np.random.seed(42)
x, y = np.random.randint(0, 256, size=2)
trace_lines, result = karatsuba_trace_demo(x, y, 8, base_bits=4)
for line in trace_lines:
    print(line)
print(f"\nVerification: {x} * {y} = {x * y} (computed: {result})")
assert result == x * y, "Karatsuba result mismatch!"

print("\n" + "=" * 80)
print("EXAMPLE 2: 16-bit x 16-bit Karatsuba trace (2 recursion levels)")
print("=" * 80)
x16 = np.random.randint(0, 2**16)
y16 = np.random.randint(0, 2**16)
trace_lines_16, result_16 = karatsuba_trace_demo(x16, y16, 16, base_bits=4)
for line in trace_lines_16:
    print(line)
print(f"\nVerification: {x16} * {y16} = {x16 * y16} (computed: {result_16})")
assert result_16 == x16 * y16

print(f"\n8-bit trace length: {len(trace_lines)} lines")
print(f"16-bit trace length: {len(trace_lines_16)} lines")

In [ ]:
# Cell 3: Compare Karatsuba vs school algorithm trace lengths

def school_trace_length(n_bits: int) -> int:
    """Estimate token count for school (long) multiplication trace.
    School multiplication of n-bit x n-bit:
    - n partial products, each ~2n bits
    - n-1 additions of ~2n bit numbers
    Total tokens ~ n * 2n + (n-1) * 2n = O(n^2)
    """
    n_partial_products = n_bits
    tokens_per_partial = 2 * n_bits + 5  # bits + delimiters
    tokens_additions = (n_bits - 1) * (2 * n_bits + 5)
    overhead = 3 * n_bits  # input/output formatting
    return n_partial_products * tokens_per_partial + tokens_additions + overhead


def karatsuba_trace_length(n_bits: int, base_bits: int = 4) -> int:
    """Estimate token count for Karatsuba trace.
    At each recursion level: 1 split + 3 sub-multiplications + 2 adds + 1 sub + 1 combine
    Each operation is O(n) tokens at that level.
    Recursion: T(n) = 3*T(n/2) + O(n)  =>  O(n^log2(3)) = O(n^1.585)
    """
    if n_bits <= base_bits:
        return 2 * n_bits + 2 * base_bits + 5  # input + output + delimiters

    half = n_bits // 2
    # Tokens for operations at this level
    split_tokens = 2 * n_bits + 10  # split operation
    add_tokens = 2 * half + 10       # addition for z1
    sub_tokens = 2 * n_bits + 10     # subtraction for z1
    combine_tokens = 2 * n_bits + 10 # final combine

    # 3 recursive sub-multiplications
    sub_mul_tokens = 3 * karatsuba_trace_length(half, base_bits)

    return split_tokens + sub_mul_tokens + add_tokens + sub_tokens + combine_tokens


# Compare trace lengths across bit widths
bit_widths = [4, 8, 16, 32, 64, 128]
school_lengths = [school_trace_length(b) for b in bit_widths]
karatsuba_lengths = [karatsuba_trace_length(b) for b in bit_widths]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].plot(bit_widths, school_lengths, 'ro-', label='School (O(n^2))', linewidth=2, markersize=8)
axes[0].plot(bit_widths, karatsuba_lengths, 'bs-', label='Karatsuba (O(n^1.585))', linewidth=2, markersize=8)
axes[0].set_xlabel('Operand bit width', fontsize=12)
axes[0].set_ylabel('Estimated trace tokens', fontsize=12)
axes[0].set_title('Trace Length Comparison (Linear Scale)', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Log-log scale (should show different slopes)
axes[1].loglog(bit_widths, school_lengths, 'ro-', label='School (O(n^2))', linewidth=2, markersize=8)
axes[1].loglog(bit_widths, karatsuba_lengths, 'bs-', label='Karatsuba (O(n^1.585))', linewidth=2, markersize=8)
axes[1].set_xlabel('Operand bit width (log)', fontsize=12)
axes[1].set_ylabel('Estimated trace tokens (log)', fontsize=12)
axes[1].set_title('Trace Length Comparison (Log-Log Scale)', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('trace_length_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print table
print(f"{'Bit Width':>10} {'School':>10} {'Karatsuba':>10} {'Ratio':>8}")
print("-" * 42)
for bw, sl, kl in zip(bit_widths, school_lengths, karatsuba_lengths):
    print(f"{bw:>10} {sl:>10} {kl:>10} {sl/kl:>8.2f}x")

print(f"\nAt 128-bit: school traces are {school_lengths[-1]/karatsuba_lengths[-1]:.1f}x longer than Karatsuba")
print(f"This means the transformer needs to process {school_lengths[-1]/karatsuba_lengths[-1]:.1f}x fewer tokens.")

In [ ]:
# Cell 4: Visualize position encodings

def generate_hierarchical_positions(n_bits: int, base_bits: int = 4):
    """Generate hierarchical position tuples for a Karatsuba trace.
    Each token gets: (bit_significance, recursion_depth, sub_problem_id, step_type)
    """
    # Step types: 0=INPUT, 1=SPLIT, 2=MULTIPLY, 3=ADD, 4=SUB, 5=COMBINE, 6=OUTPUT
    positions = []

    def add_positions(n, depth, sub_id):
        # Input tokens
        for bit in range(n):
            positions.append((bit, depth, sub_id, 0))  # INPUT
        for bit in range(n):
            positions.append((bit, depth, sub_id, 0))  # INPUT (second operand)

        if n <= base_bits:
            # Base case: direct multiply
            for bit in range(2 * n):
                positions.append((bit, depth, sub_id, 2))  # MULTIPLY (result)
            return

        half = n // 2

        # SPLIT
        for bit in range(n):
            positions.append((bit, depth, sub_id, 1))  # SPLIT

        # Three sub-multiplications
        add_positions(half, depth + 1, sub_id * 3 + 0)  # z0
        add_positions(half, depth + 1, sub_id * 3 + 1)  # z2
        add_positions(half + 1, depth + 1, sub_id * 3 + 2)  # z1 (extra bit for sum)

        # ADD, SUB, COMBINE
        for bit in range(half + 1):
            positions.append((bit, depth, sub_id, 3))  # ADD
        for bit in range(2 * n):
            positions.append((bit, depth, sub_id, 4))  # SUB
        for bit in range(2 * n):
            positions.append((bit, depth, sub_id, 5))  # COMBINE

    add_positions(n_bits, 0, 0)
    return positions


# Generate positions for 8-bit Karatsuba trace
positions_8 = generate_hierarchical_positions(8, base_bits=4)
positions_16 = generate_hierarchical_positions(16, base_bits=4)

# Extract components
bit_sigs = [p[0] for p in positions_8]
rec_depths = [p[1] for p in positions_8]
sub_ids = [p[2] for p in positions_8]
step_types = [p[3] for p in positions_8]

step_names = ['INPUT', 'SPLIT', 'MULTIPLY', 'ADD', 'SUB', 'COMBINE', 'OUTPUT']

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

token_idx = range(len(positions_8))

axes[0].scatter(token_idx, bit_sigs, c=bit_sigs, cmap='viridis', s=8, alpha=0.7)
axes[0].set_ylabel('Bit Significance', fontsize=11)
axes[0].set_title('Hierarchical Position Encoding Components (8-bit Karatsuba Trace)', fontsize=13)

axes[1].scatter(token_idx, rec_depths, c=rec_depths, cmap='Reds', s=8, alpha=0.7)
axes[1].set_ylabel('Recursion Depth', fontsize=11)

axes[2].scatter(token_idx, sub_ids, c=sub_ids, cmap='Set1', s=8, alpha=0.7)
axes[2].set_ylabel('Sub-problem ID', fontsize=11)

scatter = axes[3].scatter(token_idx, step_types, c=step_types, cmap='tab10', s=8, alpha=0.7)
axes[3].set_ylabel('Step Type', fontsize=11)
axes[3].set_xlabel('Token Position in Trace', fontsize=11)
axes[3].set_yticks(range(len(step_names)))
axes[3].set_yticklabels(step_names, fontsize=9)

for ax in axes:
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('position_encodings_8bit.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Total tokens in 8-bit trace: {len(positions_8)}")
print(f"Total tokens in 16-bit trace: {len(positions_16)}")
print(f"Ratio: {len(positions_16) / len(positions_8):.2f}x (expected ~3x for one more recursion level)")

In [ ]:
# Cell 5: Token statistics

# Analyse the distribution of tokens in Karatsuba vs school traces

def count_operations_karatsuba(n_bits: int, base_bits: int = 4) -> dict:
    """Count the number of each operation type in a Karatsuba trace."""
    counts = Counter()
    
    def count_recursive(n, depth):
        if n <= base_bits:
            counts['BASE_MUL'] += 1
            return
        half = n // 2
        counts['SPLIT'] += 1
        counts['ADD'] += 1       # x_lo + x_hi, y_lo + y_hi
        counts['SUB'] += 1       # z1 = product - z0 - z2
        counts['COMBINE'] += 1
        # 3 sub-multiplications
        count_recursive(half, depth + 1)  # z0
        count_recursive(half, depth + 1)  # z2
        count_recursive(half + 1, depth + 1)  # z1 (extra bit)
    
    count_recursive(n_bits, 0)
    return dict(counts)


def count_operations_school(n_bits: int) -> dict:
    """Count operations in school multiplication trace."""
    return {
        'PARTIAL_PRODUCT': n_bits,      # One per bit of second operand
        'SHIFT': n_bits,                # Shift each partial product
        'ADD': n_bits - 1,              # Sum all partial products
    }


print("Operation Counts: Karatsuba vs School")
print("=" * 60)

for n_bits in [8, 16, 32, 64]:
    k_ops = count_operations_karatsuba(n_bits)
    s_ops = count_operations_school(n_bits)
    
    print(f"\n--- {n_bits}-bit x {n_bits}-bit ---")
    print(f"  Karatsuba: {k_ops}")
    print(f"  School:    {s_ops}")
    print(f"  Karatsuba base muls: {k_ops.get('BASE_MUL', 0)}")
    print(f"  School partial products: {s_ops.get('PARTIAL_PRODUCT', 0)}")
    print(f"  Karatsuba total ops: {sum(k_ops.values())}")
    print(f"  School total ops: {sum(s_ops.values())}")

# Visualize operation counts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bit_widths_test = [8, 16, 32, 64, 128]

k_base_muls = [count_operations_karatsuba(b).get('BASE_MUL', 0) for b in bit_widths_test]
s_partial_prods = [count_operations_school(b).get('PARTIAL_PRODUCT', 0) for b in bit_widths_test]

axes[0].semilogy(bit_widths_test, k_base_muls, 'bs-', label='Karatsuba base multiplications', linewidth=2, markersize=8)
axes[0].semilogy(bit_widths_test, s_partial_prods, 'ro-', label='School partial products', linewidth=2, markersize=8)
axes[0].set_xlabel('Operand bit width', fontsize=12)
axes[0].set_ylabel('Count (log scale)', fontsize=12)
axes[0].set_title('Multiplication Sub-operations', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Recursion depth
import math
rec_depths_list = [max(0, int(math.log2(b / 4))) for b in bit_widths_test]
axes[1].bar(range(len(bit_widths_test)), rec_depths_list, color='steelblue', alpha=0.8)
axes[1].set_xticks(range(len(bit_widths_test)))
axes[1].set_xticklabels([str(b) for b in bit_widths_test])
axes[1].set_xlabel('Operand bit width', fontsize=12)
axes[1].set_ylabel('Karatsuba recursion depth', fontsize=12)
axes[1].set_title('Recursion Depth (base case = 4 bits)', fontsize=14)
for i, d in enumerate(rec_depths_list):
    axes[1].text(i, d + 0.05, str(d), ha='center', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('operation_counts.png', dpi=150, bbox_inches='tight')
plt.show()

# Token vocabulary analysis
print("\nToken Vocabulary:")
print("  Binary digits: 0, 1")
print("  Special tokens: [PAD], [BOS], [EOS], [SEP]")
print("  Operation markers: [SPLIT], [ADD], [SUB], [COMBINE], [MUL]")
print(f"  Total vocab size: ~10-12 tokens")
print(f"  (Compare: decimal would need 0-9 = 10 digit tokens)")